Task A Stage 1.5：自由文本评分 v3

从已核验的 v2 延续。按用户最新授权允许语义相同的完整重复最终声明。保留 raw_response，增加歧义/泄漏及分量统计。Stage 1 与 v2 文件原样保留。

In [ ]:
import re

SCORING_VERSION = 'task-a-free-text-v3'
_INT = r'[+-]?[0-9]{1,64}'
_PAIR = rf'\(\s*({_INT})\s*,\s*({_INT})\s*\)'
_LABELS = rf'A\s*=\s*({_INT})\s*[,;]\s*B\s*=\s*({_INT})'
_MARKER = re.compile(r'^(?:Final answer|Final values|Answer|最终答案|最终值)(?:\s*[:：]\s*(.*))?$', re.I)
_DECLARATION = re.compile(r'^(?:The final values are|The final answer is)\s+(.+)$', re.I)


def _normalize_line(line):
    # 只在解析副本中规范化；原始响应由评分记录原样保留。
    line = line.strip().replace('\u2212', '-')
    line = re.sub(r'^#{1,6}\s+', '', line)
    if line.startswith('**') and line.endswith('**') and line.count('**') == 2 and len(line) > 4:
        line = line[2:-2].strip()
    line = re.sub(r'\*\*([^*]+)\*\*', r'\1', line)
    if line.startswith('`') and line.endswith('`') and not line.startswith('```'):
        line = line[1:-1].strip()
    return line


def _pair_body(body):
    body = _normalize_line(body)
    if body.endswith('.'):
        body = body[:-1].rstrip()
    for pattern in (_PAIR, _LABELS):
        match = re.fullmatch(pattern, body, re.I)
        if match:
            return [int(match[1]), int(match[2])]
    return None


def parse_final_answer(raw_response, finish_reason=None):
    """v3：相同完整最终声明去重，冲突或畸形声明不回退；与 gold 无关。"""
    if not isinstance(raw_response, str):
        raise TypeError('raw_response must be a string; use empty text for empty completions')
    if finish_reason not in (None, 'stop', 'length', 'abort', 'error'):
        raise ValueError('unsupported finish_reason; map provider metadata explicitly')
    items = [(i, _normalize_line(line)) for i,line in enumerate(raw_response.splitlines()) if line.strip()]
    def fail(reason, ambiguous=False):
        return {'answer_parse_ok':False, 'parsed_answer':None, 'parse_reason':reason,
                'answer_line_indices':[], 'ambiguity':ambiguous, 'final_declaration_count':0}
    def success(value, indices, reason, count):
        return {'answer_parse_ok':True, 'parsed_answer':value, 'parse_reason':reason,
                'answer_line_indices':indices, 'ambiguity':False, 'final_declaration_count':count}
    if not items:
        return fail('empty')
    if any('```' in line for _,line in items):
        return fail('unsupported_code_fence')
    finals=[]
    for pos,(_,line) in enumerate(items):
        match=_MARKER.fullmatch(line) or _DECLARATION.fullmatch(line)
        if match:
            finals.append((pos,match[1]))
        elif re.search(r'\b(?:final answer|final values|answer is)\b|最终答案|最终值',line,re.I):
            return fail('unsupported_final_declaration')
    if finals:
        values, indices, consumed = [], [], set()
        for k,(pos,body) in enumerate(finals):
            boundary=finals[k+1][0] if k+1<len(finals) else len(items)
            span=[pos]
            if body:
                value=_pair_body(body)
            else:
                following=items[pos+1:boundary]
                value=_pair_body(following[0][1]) if following else None
                if value is not None:
                    span.append(pos+1)
                elif len(following)>=2:
                    value=_pair_body(following[0][1]+', '+following[1][1])
                    if value is not None:span.extend([pos+1,pos+2])
            if value is None:
                # 显式多个候选即歧义；缺项/半截答案为语法失败，均不得回退。
                alternatives=len(re.findall(_PAIR,body))>1 or bool(re.search(r'\bor\b|或者|或是',body,re.I))
                return fail('ambiguous_final_alternatives' if alternatives else 'malformed_final_answer', alternatives)
            values.append(value)
            consumed.update(span)
            indices.extend(items[p][0] for p in span)
        if len({tuple(v) for v in values})!=1:
            return fail('conflicting_final_declarations',True)
        if any(_pair_body(line) is not None for pos,(_,line) in enumerate(items)
               if pos not in consumed):
            return fail('unlabelled_competing_answer',True)
        # 声明之后的额外材料可能是在撤回或改写答案。只允许 length 时的无数字、
        # 无不确定/否定措辞的说明尾段；其余保守拒绝，防止截断造成误判。
        last=max(consumed)
        tail=[line for pos,(_,line) in enumerate(items) if pos>last]
        if tail:
            text=' '.join(tail)
            safe_tail=(finish_reason=='length' and not re.search(
                r'[0-9=()]|\b(?:not|wrong|incorrect|actually|instead|maybe|perhaps|wait|or)\b|错|不|或',text,re.I))
            if not safe_tail:return fail('text_after_final_answer')
        return success(values[0],indices,'explicit_final' if len(values)==1 else 'identical_final_declarations',len(values))
    if len(items)==1:
        value=_pair_body(items[0][1])
    elif len(items)==2:
        value=_pair_body(', '.join(line for _,line in items))
    else:
        value=None
    if value is not None:
        if finish_reason in ('length','abort','error'):
            return fail('unfinished_unmarked_answer')
        return success(value,[i for i,_ in items],'answer_only',0)
    bare_candidates=[_pair_body(line) for _,line in items if _pair_body(line) is not None]
    if len(bare_candidates)>1 or (len(items)==1 and len(re.findall(_PAIR,items[0][1]))>1):
        return fail('multiple_unmarked_candidates',True)
    return fail('no_unambiguous_final_answer')


def _da_diagnostics(raw_response, parsed, mode):
    if mode != 'DA':
        return {'da_extra_text': None, 'da_arith': None, 'da_trace': None,
                'da_trace_evidence_count': None}
    answer_lines = set(parsed['answer_line_indices'])
    residual = [line.strip() for i, line in enumerate(raw_response.splitlines())
                if i not in answer_lines and line.strip()]
    # 即使最终声明冲突，也不能把声明本身误作推理泄漏证据。
    # 这里排除的是诊断候选行，不改变 parse_ok 或主评分。
    source = [(i, _normalize_line(line)) for i,line in enumerate(raw_response.splitlines()) if line.strip()]
    candidate_lines = set(answer_lines)
    for pos,(i,line) in enumerate(source):
        declaration = _MARKER.fullmatch(line) or _DECLARATION.fullmatch(line)
        if declaration:
            candidate_lines.add(i)
            if not declaration[1]:
                following = source[pos+1:pos+3]
                if following and _pair_body(following[0][1]) is not None:
                    candidate_lines.add(following[0][0])
                elif len(following)==2 and _pair_body(following[0][1]+', '+following[1][1]) is not None:
                    candidate_lines.update(j for j,_ in following)
    reasoning = [line.strip() for i,line in enumerate(raw_response.splitlines())
                 if i not in candidate_lines and line.strip()]
    # 启发式诊断，不作为最终答案准确率的过滤条件。
    def arithmetic(line):
        numeric = re.search(r'[0-9]', line)
        return bool(numeric and (
            re.search(r'[0-9]\s*[+\-−*/×÷]\s*[+\-−]?[0-9]', line)
            or re.search(r'\b(?:add|subtract|plus|minus|multiply|divide)\b|加上|减去|乘以|除以', line, re.I)
            or re.search(r'\b[AB]\s*[+\-]=\s*[0-9]', line, re.I)))
    def state_claim(line):
        return bool(re.search(r'\b[AB]\s*=\s*[+\-−]?[0-9]', line, re.I)
                    or re.search(_PAIR, line.replace('−','-'))
                    or re.search(r'(?:\bstep\s*[0-9]+|第\s*[0-9]+\s*步)', line, re.I))
    evidence_lines = sum(arithmetic(line) or state_claim(line) for line in reasoning)
    step_labels = set(re.findall(r'\bstep\s*([0-9]+)|第\s*([0-9]+)\s*步', '\n'.join(reasoning), re.I))
    evidence = max(evidence_lines, len(step_labels))
    return {'da_extra_text': bool(residual), 'da_arith': any(arithmetic(line) for line in reasoning),
            'da_trace': evidence >= 2, 'da_trace_evidence_count': evidence}


def score_free_text(raw_response, gold, mode, finish_reason=None):
    if mode not in ('DA', 'CoT'):
        raise ValueError('mode must be DA or CoT')
    if not isinstance(gold, (tuple,list)) or len(gold) != 2 or any(type(x) is not int for x in gold):
        raise ValueError('gold must be an ordered integer pair')
    parsed = parse_final_answer(raw_response, finish_reason)
    answer_ok = parsed['answer_parse_ok'] and parsed['parsed_answer'] == list(gold)
    diagnostics = _da_diagnostics(raw_response, parsed, mode)
    return {'scoring_version': SCORING_VERSION, 'raw_response': raw_response,
            'mode': mode, 'finish_reason': finish_reason, **parsed,
            'answer_ok': answer_ok, 'final_answer_ok': answer_ok,
            'a_ok': parsed['answer_parse_ok'] and parsed['parsed_answer'][0] == gold[0],
            'b_ok': parsed['answer_parse_ok'] and parsed['parsed_answer'][1] == gold[1],
            'da_leakage': (diagnostics['da_arith'] or diagnostics['da_trace']) if mode == 'DA' else None,
            'parse_failure': not parsed['answer_parse_ok'],
            'truncation': None if finish_reason is None else finish_reason == 'length',
            'truncated': None if finish_reason is None else finish_reason == 'length',
            'runtime_failure': finish_reason in ('abort', 'error'),
            **diagnostics}


def summarize_scores(records):
    """主分母包含全部记录；诊断同时给出分子/分母，不静默排除失败。"""
    if any(r['scoring_version']!=SCORING_VERSION for r in records):
        raise ValueError('do not mix scoring versions')
    n=len(records)
    parsed=sum(r['answer_parse_ok'] for r in records)
    correct=sum(r['answer_ok'] for r in records)
    rate=lambda a,b: a/b if b else None
    known=[r for r in records if r['truncation'] is not None]
    da=[r for r in records if r['mode']=='DA']
    out={'n_total':n,'n_parsed':parsed,'n_correct':correct,
         'primary_answer_accuracy':rate(correct,n),'main_accuracy':rate(correct,n),
         'parse_success_rate':rate(parsed,n),'parse_failure_count':n-parsed,
         'parse_failure_rate':rate(n-parsed,n),'conditional_answer_accuracy':rate(correct,parsed),
         'ambiguity_count':sum(r['ambiguity'] for r in records),
         'ambiguity_rate':rate(sum(r['ambiguity'] for r in records),n),
         'truncation_known_n':len(known),'truncation_unknown_n':n-len(known),
         'truncation_count':sum(r['truncation'] for r in known),
         'truncation_rate':rate(sum(r['truncation'] for r in known),len(known)),
         'runtime_failure_count':sum(r['runtime_failure'] for r in records),
         'da_n':len(da)}
    for name in ('da_extra_text','da_arith','da_trace','da_leakage'):
        numerator=sum(r[name] for r in da)
        out[name+'_count']=numerator
        out[name+'_rate']=rate(numerator,len(da))
    for component in ('a','b'):
        numerator=sum(r[component+'_ok'] for r in records)
        out[component+'_correct_count']=numerator
        out[component+'_accuracy']=rate(numerator,n)
        out[component+'_conditional_accuracy']=rate(numerator,parsed)
    return out
